# Email Dataset Case Study
EDAP (Exploratory Data Analysis with Python) — analyzing Gmail export (mbox) for sent/received patterns, timing behavior, and subject word cloud.

## 1. Load the mbox file and export to CSV
Download your Gmail data via Google Takeout, upload the `.mbox` file to Colab's `/content/` folder, and update the filename below if needed.

In [ ]:
import mailbox as mb

mboxfile = "/content/All mail Including Spam and Trash.mbox"
mbox = mb.mbox(mboxfile)
mbox

In [ ]:
# Inspect available header fields on the first message
for key in mbox[0].keys():
    print(key)

In [ ]:
import csv

with open("mailbox.csv", "w", newline="", encoding="utf-8") as outputfile:
    writer = csv.writer(outputfile)

    writer.writerow([
        "subject",
        "from",
        "date",
        "to",
        "label",
        "thread"
    ])

    for message in mbox:
        writer.writerow([
            message["subject"],
            message["from"],
            message["date"],
            message["to"],
            message["X-Gmail-Labels"],
            message["X-GM-THRID"]
        ])

## 2. Load into pandas and clean up

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("mailbox.csv")
df.head()

In [ ]:
df.dtypes

In [ ]:
df['date'] = df['date'].apply(
    lambda x: pd.to_datetime(x, errors='coerce', utc=True)
)
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
# Drop rows where the date couldn't be parsed
df = df[df['date'].notna()]
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.to_csv('gmail.csv')
df.info()

In [ ]:
df.head(10)

## 3. Extract clean email addresses from the `from` field

In [ ]:
import re

def extract_email_ID(string):
    email = re.findall(r'<(.+?)>', string)
    if not email:
        email = list(filter(lambda y: '@' in y, string.split()))
    return email[0] if email else np.nan

df['from'] = df['from'].apply(lambda x: extract_email_ID(x))
df.head()

## 4. Label each email as sent (by me) or inbox (received)
Set `myemail` to the account the mbox was exported from.

In [ ]:
myemail = "saiayyappa81@gmail.com"

df['label'] = df['from'].apply(
    lambda x: 'sent' if x == myemail else 'inbox'
)

## 5. Timezone conversion and derived time features

In [ ]:
import datetime
import pytz

def refactor_timezone(x):
    est = pytz.timezone('US/Eastern')
    return x.astimezone(est)

df['date'] = df['date'].apply(lambda x: refactor_timezone(x))

In [ ]:
df['dayofweek'] = df['date'].apply(lambda x: x.day_name())
df['timeofday'] = df['date'].apply(lambda x: x.hour + x.minute / 60 + x.second / 3600)
df['hour'] = df['date'].apply(lambda x: x.hour)
df['year_int'] = df['date'].apply(lambda x: x.year)
df['year'] = df['date'].apply(lambda x: x.year + x.dayofyear / 365.25)

df.index = df['date']
del df['date']
df.head()

In [ ]:
print(df.index.min().strftime("%a, %d %b %Y %I:%M %p"))
print(df.index.max().strftime("%a, %d %b %Y %I:%M %p"))
print(df['label'].value_counts())

## 6. Sent vs. Received — time of day across years (scatter)

In [ ]:
sent = df[df["label"] == "sent"]
received = df[df["label"] == "inbox"]

def plot_todo_vs_year(data, ax, color="steelblue", title=""):
    ax.scatter(
        data["year"],
        data["timeofday"],
        s=6,
        alpha=0.6,
        color=color
    )
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Time of Day")
    ax.set_ylim(0, 24)
    ax.set_yticks(range(0, 25, 3))
    ax.set_yticklabels([
        "12 AM", "03 AM", "06 AM", "09 AM", "12 PM",
        "03 PM", "06 PM", "09 PM", "12 AM"
    ])
    ax.grid(ls=":", color="gray")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

plot_todo_vs_year(
    sent,
    ax[0],
    color="blue",
    title="Sent Emails"
)

plot_todo_vs_year(
    received,
    ax[1],
    color="red",
    title="Received Emails"
)

plt.tight_layout()
plt.show()

## 7. Average emails per day and per hour

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
from scipy import ndimage
from scipy.interpolate import interp1d
import datetime

# ------------------------------------------------------------
# Create Sent and Received DataFrames
# ------------------------------------------------------------
df["label"] = df["label"].astype(str)
sent = df[df["label"].str.contains("Sent", case=False, na=False)]
received = df[df["label"].str.contains("Inbox", case=False, na=False)]

# ------------------------------------------------------------
# Function 1 : Average Emails Per Day
# ------------------------------------------------------------
def plot_number_perday_per_year(df, ax, label=None, dt=0.3, **plot_kwargs):

    year = df[df["year"].notna()]["year"].values
    if len(year) == 0:
        return

    Ty = year.max() - year.min()
    bins = int(Ty / dt)

    # Ensure bins is at least 1 if there is data
    if bins < 1:
        bins = 1

    weights = 1 / (np.ones_like(year) * dt * 365.25)

    ax.hist(
        year,
        bins=bins,
        weights=weights,
        label=label,
        **plot_kwargs
    )

    ax.grid(ls=":", color="gray")

# ------------------------------------------------------------
# Function 2 : Average Emails Per Hour
# ------------------------------------------------------------
def plot_number_perhour_per_year(df, ax, label=None, dt=1,
                                  smooth=False, weight_fun=None,
                                  **plot_kwargs):

    tod = df[df["timeofday"].notna()]["timeofday"].values
    year = df[df["timeofday"].notna()]["year"].values
    if len(tod) == 0:
        return

    Ty = year.max() - year.min()
    T = tod.max() - tod.min()
    bins = int(T / dt)
    if bins < 1:
        bins = 1

    if weight_fun is None:
        weights = 1 / (np.ones_like(tod) * Ty * 365.25 / dt)
    else:
        weights = weight_fun(df)

    if smooth:
        hist, edges = np.histogram(tod, bins=bins, weights=weights)
        x = np.delete(edges, -1) + 0.5 * (edges[1] - edges[0])
        hist = ndimage.gaussian_filter(hist, sigma=0.75)
        f = interp1d(x, hist, kind="cubic")
        xx = np.linspace(x.min(), x.max(), 1000)
        ax.plot(xx, f(xx), label=label, **plot_kwargs)
    else:
        ax.hist(
            tod, bins=bins, weights=weights,
            label=label, orientation="horizontal", **plot_kwargs
        )

    ax.grid(ls=":", color="gray")

# ------------------------------------------------------------
# Combined layout: top = emails/day, bottom-left = scatter,
# bottom-right = emails/hour
# ------------------------------------------------------------
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                        hspace=0.05, wspace=0.05)

ax_top = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0], sharex=ax_top)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

plot_number_perday_per_year(received, ax_top, color="royalblue", label="Incoming")
plot_number_perday_per_year(sent, ax_top, color="orange", label="Outgoing")
ax_top.set_ylabel("Average Emails per Day")
ax_top.legend(loc="upper left")
plt.setp(ax_top.get_xticklabels(), visible=False)

ax_main.scatter(received["year"], received["timeofday"], s=6, alpha=0.6, color="royalblue")
ax_main.set_xlabel("Year")
ax_main.set_ylabel("Time of Day")
ax_main.set_ylim(0, 24)
ax_main.set_yticks(range(0, 25, 3))
ax_main.set_yticklabels([
    "12 AM", "03 AM", "06 AM", "09 AM", "12 PM",
    "03 PM", "06 PM", "09 PM", "12 AM"
])
ax_main.grid(ls=":", color="gray")

plot_number_perhour_per_year(received, ax_right, color="royalblue")
ax_right.set_xlabel("Average Emails per Hour")
plt.setp(ax_right.get_yticklabels(), visible=False)

plt.show()

## 8. Emails per day of week

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Count total emails for each day of the week
# ------------------------------------------------------------
counts = df["dayofweek"].value_counts(sort=False)

print("Number of Emails Per Day")
print(counts)

# ------------------------------------------------------------
# Plot Bar Chart
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))

counts.plot(
    kind="bar",
    color="steelblue",
    edgecolor="black"
)

plt.title("Number of Emails per Day")
plt.xlabel("Day of Week")
plt.ylabel("Number of Emails")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Incoming vs Outgoing fraction, per day of week
# ------------------------------------------------------------
weekly = pd.crosstab(df["dayofweek"], df["label"], normalize="columns")
weekly = weekly.rename(columns={"sent": "Outgoing Email", "inbox": "Incoming Email"})

print("Weekly Email Fraction")
print(weekly)

weekly.plot(
    kind="bar",
    figsize=(9, 5),
    color=["steelblue", "orange"],
    edgecolor="black"
)

plt.title("Incoming vs Outgoing Emails per Day")
plt.xlabel("Day of Week")
plt.ylabel("Fraction of Weekly Emails")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Emails per hour, for each day of the week

In [ ]:
# ------------------------------------------------------------
# Emails Per Hour for Each Day of the Week (Single Cell)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

for day, group in df.groupby("dayofweek"):
    plot_number_perhour_per_year(
        group, ax, label=day, dt=1, smooth=True
    )

ax.set_xlabel("Time of Day")
ax.set_ylabel("Fraction of Weekly Emails per Hour", fontsize=12)
ax.set_title("Average Emails Per Hour for Each Day", fontsize=15)

ax.set_xticks(range(0, 25, 3))
ax.set_xticklabels([
    "12 AM", "03 AM", "06 AM", "09 AM", "12 PM",
    "03 PM", "06 PM", "09 PM", "12 AM"
])

plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 10. Word cloud of email subjects

In [ ]:
# Install the library (Run once in Google Colab)
!pip -q install wordcloud

In [ ]:
# Import libraries
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

# ------------------------------------------------------------
# Remove unwanted sender
# ------------------------------------------------------------
df_no_arxiv = df[df["from"].astype(str) != "no-reply@arXiv.org"]

# ------------------------------------------------------------
# Combine all email subjects into one string
# ------------------------------------------------------------
text = " ".join(df_no_arxiv["subject"].dropna().astype(str))

# ------------------------------------------------------------
# Add custom stop words
# ------------------------------------------------------------
stopwords = set(STOPWORDS)

custom_words = [
    "Re",
    "Fwd",
    "FW",
    "RE",
]

stopwords.update(custom_words)

# ------------------------------------------------------------
# Create Word Cloud
# ------------------------------------------------------------
wordcloud = WordCloud(
    width=800,
    height=500,
    background_color="white",
    stopwords=stopwords,
    collocations=False
).generate(text)

# ------------------------------------------------------------
# Display Word Cloud
# ------------------------------------------------------------
plt.figure(figsize=(14, 8))

plt.imshow(wordcloud, interpolation="bilinear")

plt.axis("off")

plt.title("Word Cloud of Email Subjects", fontsize=18)

plt.tight_layout()

plt.show()